In [1]:
print("hello world")

hello world


In [3]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


import vitaldb as v

In [4]:
track_names = ["SNUADC/ART",        # Arterial pressure wave  | W/500 | mmHg
               "SNUADC/ECG_II",     # ECG lead II wave        | W/500 | mV
               "SNUADC/ECG_V5",     # ECG lead V5 wave        | W/500 | mV
               "BIS/EEG1_WAV",      # EEG wave from channel 1 | W/128 | uV
               "BIS/EEG2_WAV",      # EEG wave from channel 2 | W/128 | uV
               "Solar8000/RR_CO2",  # Respiratory rate based on capnography | N | /min
               "Primus/CO2",        # Capnography wave        | W/62.5 | mmHg
               "BIS/BIS",           # Bispectral index value  |    N   | unitless
               ]                

# read 1 patient from the VitalDB server
patient = v.VitalFile(1, track_names)  # should we add max length?
# convert to pandas dataframe
patient = patient.to_pandas(track_names=track_names, interval=5)
# change column names to 'human-readable'
patient.columns = ['arterial_pres', 'ecg1', 'ecg2', 'eeg1', 'eeg2', 'resp_rate', 'capnography', 'bis']

In [7]:
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(TimeSeriesLSTM, self).__init__()

        # LSTM Layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully-connected Output Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x) # _ : (hn, cn) are the final states

        out = out[:, -1, :]

        prediction = self.fc(out)
        return prediction
        

In [9]:
epochs = 20
batch_size = 32
seq_len = 200
split = 0.8
# device = 'cuda'
device = 'cpu'

patient = patient[['eeg1', 'eeg2', 'bis']].dropna()
patient = patient[patient['bis'] > 0]
patient['eeg1'] = (patient['eeg1'] - patient['eeg1'].mean()) / patient['eeg1'].std()
patient['eeg2'] = (patient['eeg2'] - patient['eeg2'].mean()) / patient['eeg2'].std()
x = np.stack([patient.eeg1, patient.eeg2], axis=1)
X = np.array([x[i:i+seq_len] for i in range(x.shape[0]-seq_len)])
X = X.reshape(-1, seq_len, 2)

y = patient.bis
y = (y - np.mean(y)) / np.std(y)
Y = np.array([y[i+seq_len] for i in range(len(y)-seq_len)]).reshape(-1, 1)

split_idx = int(len(X) * split)
X_train = X[:split_idx]
Y_train = Y[:split_idx]

model = TimeSeriesLSTM(2, 32, 3)
model.to(device)
model.train()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

dataset_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32).to(device), 
                        torch.tensor(Y_train, dtype=torch.float32).to(device))
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)

X_val = X[split_idx:]
Y_val = Y[split_idx:]
dataset_val = TensorDataset(torch.tensor(X_val, dtype=torch.float32).to(device), 
                            torch.tensor(Y_val, dtype=torch.float32).to(device))
dataloader_val = DataLoader(dataset_val, batch_size=batch_size)

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in dataloader_train:
        optimizer.zero_grad()
        y_hat = model(batch_x)
        loss = criterion(y_hat, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Check
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for v_x, v_y in dataloader_val:
            v_hat = model(v_x)
            v_loss = criterion(v_hat, v_y)
            val_loss += v_loss.item()
    
    print(f"Epoch {epoch+1}: Train Loss {train_loss/len(dataloader_train):.4f} | Val Loss {val_loss/len(dataloader_val):.4f}")

Epoch 1: Train Loss 0.7025 | Val Loss 1.2324
Epoch 2: Train Loss 0.6853 | Val Loss 1.2250
Epoch 3: Train Loss 0.6608 | Val Loss 1.3095
Epoch 4: Train Loss 0.5515 | Val Loss 1.8141
Epoch 5: Train Loss 0.4820 | Val Loss 2.2224
Epoch 6: Train Loss 0.4743 | Val Loss 1.6707
Epoch 7: Train Loss 0.4658 | Val Loss 2.4477
Epoch 8: Train Loss 0.4992 | Val Loss 2.3799
Epoch 9: Train Loss 0.4707 | Val Loss 2.3834
Epoch 10: Train Loss 0.4714 | Val Loss 2.3491
Epoch 11: Train Loss 0.4761 | Val Loss 1.5318
Epoch 12: Train Loss 0.4985 | Val Loss 2.4078
Epoch 13: Train Loss 0.4660 | Val Loss 2.3034
Epoch 14: Train Loss 0.4585 | Val Loss 1.6707
Epoch 15: Train Loss 0.4609 | Val Loss 1.6545
Epoch 16: Train Loss 0.4550 | Val Loss 2.4310
Epoch 17: Train Loss 0.4676 | Val Loss 1.7188
Epoch 18: Train Loss 0.4586 | Val Loss 2.5137
Epoch 19: Train Loss 0.4749 | Val Loss 1.6840
Epoch 20: Train Loss 0.4600 | Val Loss 2.3602
